In [1]:
%load_ext autoreload
%autoreload 2
import pandas as pd

from boh.data import apply_champion_labels
from boh.data import load_raw

raw = load_raw()
constructor_df = raw["constructor_finances"]
business_df = raw["f1_business"]

In [2]:
constructor_df

,season,team_name,country_iso3,operating_budget_usd_m,sponsorship_revenue_usd_m,prize_money_usd_m,total_revenue_usd_m,in_cost_cap_era,cost_cap_limit_usd_m,is_constructors_champion
0,2010,Mercedes,DEU,270.0,86.2,112.6,198.8,0,NaN,0
1,2011,Mercedes,DEU,320.5,100.2,118.4,218.6,0,NaN,0
2,2012,Mercedes,DEU,308.3,102.4,130.4,232.9,0,NaN,0
3,2013,Mercedes,DEU,356.9,126.9,158.6,285.6,0,NaN,0
4,2014,Mercedes,DEU,339.4,128.8,173.6,302.3,0,NaN,1
...,...,...,...,...,...,...,...,...,...,...
159,2022,Haas,USA,125.7,86.5,51.6,138.1,1,140.0,0
160,2023,Haas,USA,140.7,103.4,53.8,157.3,1,135.0,0
161,2024,Haas,USA,147.2,118.8,62.6,181.4,1,135.0,0
162,2025,Haas,USA,150.5,131.6,80.1,211.8,1,135.0,0


In [3]:
business_df

,season,total_revenue_usd_m,race_promotion_fees_usd_m,broadcast_media_usd_m,sponsorship_usd_m,other_revenue_usd_m,is_liberty_media_era,drive_to_survive_viewers_m_est,us_avg_race_viewers_m,global_avg_race_viewers_m
0,2010,1108.0,399.0,377.0,177.0,155.0,0,0,0.55,70.0
1,2011,1265.0,455.0,430.0,202.0,177.0,0,0,0.55,71.5
2,2012,1377.0,496.0,468.0,220.0,193.0,0,0,0.55,73.0
3,2013,1509.0,543.0,513.0,241.0,211.0,0,0,0.55,74.5
4,2014,1728.0,622.0,588.0,276.0,242.0,0,0,0.55,76.0
5,2015,1731.0,623.0,588.0,277.0,242.0,0,0,0.55,77.5
6,2016,1805.0,650.0,614.0,289.0,253.0,0,0,0.55,79.0
7,2017,1835.0,660.0,624.0,294.0,257.0,1,0,0.55,80.5
8,2018,1806.0,650.0,614.0,289.0,253.0,1,0,0.55,82.0
9,2019,2022.0,728.0,688.0,324.0,283.0,1,0,0.60,83.5


In [4]:
constructor_df_copy = constructor_df.copy()
constructor_df_copy

,season,team_name,country_iso3,operating_budget_usd_m,sponsorship_revenue_usd_m,prize_money_usd_m,total_revenue_usd_m,in_cost_cap_era,cost_cap_limit_usd_m,is_constructors_champion
0,2010,Mercedes,DEU,270.0,86.2,112.6,198.8,0,NaN,0
1,2011,Mercedes,DEU,320.5,100.2,118.4,218.6,0,NaN,0
2,2012,Mercedes,DEU,308.3,102.4,130.4,232.9,0,NaN,0
3,2013,Mercedes,DEU,356.9,126.9,158.6,285.6,0,NaN,0
4,2014,Mercedes,DEU,339.4,128.8,173.6,302.3,0,NaN,1
...,...,...,...,...,...,...,...,...,...,...
159,2022,Haas,USA,125.7,86.5,51.6,138.1,1,140.0,0
160,2023,Haas,USA,140.7,103.4,53.8,157.3,1,135.0,0
161,2024,Haas,USA,147.2,118.8,62.6,181.4,1,135.0,0
162,2025,Haas,USA,150.5,131.6,80.1,211.8,1,135.0,0


In [5]:
before = constructor_df_copy.groupby("season")["is_constructors_champion"].sum()
corrected_df = apply_champion_labels(constructor_df_copy)
after = corrected_df.groupby("season")["is_constructors_champion"].sum(min_count=1)
pd.DataFrame({"champions in source": before, "after correction": after})

,champions in source,after correction
season,,
2010,0,1.0
2011,0,1.0
2012,0,1.0
2013,0,1.0
2014,1,1.0
2015,1,1.0
2016,1,1.0
2017,1,1.0
2018,1,1.0


In [6]:
champions = corrected_df.loc[corrected_df["is_constructors_champion"] > 0, ["season", "team_name"]].sort_values("season")
champions


,season,team_name
17,2010,Red Bull Racing
18,2011,Red Bull Racing
19,2012,Red Bull Racing
20,2013,Red Bull Racing
4,2014,Mercedes
5,2015,Mercedes
6,2016,Mercedes
7,2017,Mercedes
8,2018,Mercedes
9,2019,Mercedes


In [7]:
from boh.data import CHAMPIONS


sample = constructor_df[constructor_df["season"].isin([2025, 2026])][["season", "team_name"]]
sample.assign(champion=sample["season"].map(CHAMPIONS))

,season,team_name,champion
15,2025,Mercedes,McLaren
16,2026,Mercedes,NaN
32,2025,Red Bull Racing,McLaren
33,2026,Red Bull Racing,NaN
49,2025,Ferrari,McLaren
50,2026,Ferrari,NaN
66,2025,McLaren,McLaren
67,2026,McLaren,NaN
83,2025,Aston Martin,McLaren
84,2026,Aston Martin,NaN


In [8]:
raw = load_raw()
cf = raw["constructor_finances"]

print(cf.shape)              # expect (164, 10)
print(cf["season"].dtype)    # expect int64

sample = cf[cf["season"].isin([2025, 2026])][["season", "team_name"]]
print(len(sample))           # expect 20: 10 teams x 2 seasons

(164, 10)
int64
20


In [10]:
step_by_step = sample.assign(
    champion=sample["season"].map(CHAMPIONS),
)
step_by_step["is_match"] = step_by_step["team_name"] == step_by_step["champion"]
step_by_step["as_float"] = step_by_step["is_match"].astype(float)

step_by_step

,season,team_name,champion,is_match,as_float
15,2025,Mercedes,McLaren,False,0.0
16,2026,Mercedes,NaN,False,0.0
32,2025,Red Bull Racing,McLaren,False,0.0
33,2026,Red Bull Racing,NaN,False,0.0
49,2025,Ferrari,McLaren,False,0.0
50,2026,Ferrari,NaN,False,0.0
66,2025,McLaren,McLaren,True,1.0
67,2026,McLaren,NaN,False,0.0
83,2025,Aston Martin,McLaren,False,0.0
84,2026,Aston Martin,NaN,False,0.0


In [14]:
f1_business_df_copy = business_df.copy()
f1_business_df_copy

,season,total_revenue_usd_m,race_promotion_fees_usd_m,broadcast_media_usd_m,sponsorship_usd_m,other_revenue_usd_m,is_liberty_media_era,drive_to_survive_viewers_m_est,us_avg_race_viewers_m,global_avg_race_viewers_m
0,2010,1108.0,399.0,377.0,177.0,155.0,0,0,0.55,70.0
1,2011,1265.0,455.0,430.0,202.0,177.0,0,0,0.55,71.5
2,2012,1377.0,496.0,468.0,220.0,193.0,0,0,0.55,73.0
3,2013,1509.0,543.0,513.0,241.0,211.0,0,0,0.55,74.5
4,2014,1728.0,622.0,588.0,276.0,242.0,0,0,0.55,76.0
5,2015,1731.0,623.0,588.0,277.0,242.0,0,0,0.55,77.5
6,2016,1805.0,650.0,614.0,289.0,253.0,0,0,0.55,79.0
7,2017,1835.0,660.0,624.0,294.0,257.0,1,0,0.55,80.5
8,2018,1806.0,650.0,614.0,289.0,253.0,1,0,0.55,82.0
9,2019,2022.0,728.0,688.0,324.0,283.0,1,0,0.60,83.5


In [29]:
race_promotion_fees = f1_business_df_copy.loc[f1_business_df_copy["season"] == 2020, 'race_promotion_fees_usd_m'].sum()
broadcast_media_usd_m = f1_business_df_copy.loc[f1_business_df_copy["season"] == 2020, 'broadcast_media_usd_m'].sum()
sponsorship_usd_m = f1_business_df_copy.loc[f1_business_df_copy["season"] == 2020, 'sponsorship_usd_m'].sum()
other_revenue_usd_m = f1_business_df_copy.loc[f1_business_df_copy["season"] == 2020, 'other_revenue_usd_m'].sum()
f1_business_df_copy.loc[f1_business_df_copy["season"] == 2020, 'total_revenue_usd_m'] = race_promotion_fees + broadcast_media_usd_m + sponsorship_usd_m + other_revenue_usd_m
f1_business_df_copy

,season,total_revenue_usd_m,race_promotion_fees_usd_m,broadcast_media_usd_m,sponsorship_usd_m,other_revenue_usd_m,is_liberty_media_era,drive_to_survive_viewers_m_est,us_avg_race_viewers_m,global_avg_race_viewers_m
0,2010,1108.0,399.0,377.0,177.0,155.0,0,0,0.55,70.0
1,2011,1265.0,455.0,430.0,202.0,177.0,0,0,0.55,71.5
2,2012,1377.0,496.0,468.0,220.0,193.0,0,0,0.55,73.0
3,2013,1509.0,543.0,513.0,241.0,211.0,0,0,0.55,74.5
4,2014,1728.0,622.0,588.0,276.0,242.0,0,0,0.55,76.0
5,2015,1731.0,623.0,588.0,277.0,242.0,0,0,0.55,77.5
6,2016,1805.0,650.0,614.0,289.0,253.0,0,0,0.55,79.0
7,2017,1835.0,660.0,624.0,294.0,257.0,1,0,0.55,80.5
8,2018,1806.0,650.0,614.0,289.0,253.0,1,0,0.55,82.0
9,2019,2022.0,728.0,688.0,324.0,283.0,1,0,0.60,83.5


In [32]:
f1_business_df_copy.groupby("season").nunique()


,total_revenue_usd_m,race_promotion_fees_usd_m,broadcast_media_usd_m,sponsorship_usd_m,other_revenue_usd_m,is_liberty_media_era,drive_to_survive_viewers_m_est,us_avg_race_viewers_m,global_avg_race_viewers_m
season,,,,,,,,,
2010,1,1,1,1,1,1,1,1,1
2011,1,1,1,1,1,1,1,1,1
2012,1,1,1,1,1,1,1,1,1
2013,1,1,1,1,1,1,1,1,1
2014,1,1,1,1,1,1,1,1,1
2015,1,1,1,1,1,1,1,1,1
2016,1,1,1,1,1,1,1,1,1
2017,1,1,1,1,1,1,1,1,1
2018,1,1,1,1,1,1,1,1,1


In [39]:
f1_business_df = f1_business_df_copy[["season", "total_revenue_usd_m"]]
f1_business_df

,season,total_revenue_usd_m
0,2010,1108.0
1,2011,1265.0
2,2012,1377.0
3,2013,1509.0
4,2014,1728.0
5,2015,1731.0
6,2016,1805.0
7,2017,1835.0
8,2018,1806.0
9,2019,2022.0
